In [4]:
from pathlib import Path
import numpy as np
import pandas as pd

# -------------------------------------------------------------------------
# 1. BASE DE REFERÊNCIA (TABELA VERDADE DO TCC ORIGINAL - TIs 31, 33 e 38)
# -------------------------------------------------------------------------
BENCHMARK_TCC = pd.DataFrame([
    # TI 31
    {"TI": 31, "EMPRESA": "CEMIG", "VOC_MEDIA_TCC": 310990.91, "VOC_MEDIANA_TCC": 4864.73, "VNR_MEDIA_TCC": 380508.58, "VNR_MEDIANA_TCC": 10644.69},
    {"TI": 31, "EMPRESA": "ENEL SP", "VOC_MEDIA_TCC": 991849.47, "VOC_MEDIANA_TCC": 228266.97, "VNR_MEDIA_TCC": 216290.10, "VNR_MEDIANA_TCC": 160210.00},
    {"TI": 31, "EMPRESA": "Equatorial Para", "VOC_MEDIA_TCC": 480809.92, "VOC_MEDIANA_TCC": 82810.88, "VNR_MEDIA_TCC": 465148.57, "VNR_MEDIANA_TCC": 99656.51},
    {"TI": 31, "EMPRESA": "CPFL Piratininga", "VOC_MEDIA_TCC": 43876.22, "VOC_MEDIANA_TCC": 43876.22, "VNR_MEDIA_TCC": 43876.22, "VNR_MEDIANA_TCC": 43876.22},
    # TI 33
    {"TI": 33, "EMPRESA": "CEMIG", "VOC_MEDIA_TCC": 613815.00, "VOC_MEDIANA_TCC": 31431.00, "VNR_MEDIA_TCC": 701596.70, "VNR_MEDIANA_TCC": 41505.00},
    {"TI": 33, "EMPRESA": "ENEL SP", "VOC_MEDIA_TCC": 737545.84, "VOC_MEDIANA_TCC": 162021.66, "VNR_MEDIA_TCC": 314487.70, "VNR_MEDIANA_TCC": 80406.00},
    {"TI": 33, "EMPRESA": "Equatorial Para", "VOC_MEDIA_TCC": 533306.00, "VOC_MEDIANA_TCC": 65862.31, "VNR_MEDIA_TCC": 604984.50, "VNR_MEDIANA_TCC": 85787.00},
    {"TI": 33, "EMPRESA": "CPFL Piratininga", "VOC_MEDIA_TCC": 445228.50, "VOC_MEDIANA_TCC": 45563.00, "VNR_MEDIA_TCC": 475472.00, "VNR_MEDIANA_TCC": 63611.20},
    # TI 38
    {"TI": 38, "EMPRESA": "CEMIG", "VOC_MEDIA_TCC": 755437.78, "VOC_MEDIANA_TCC": 282504.71, "VNR_MEDIA_TCC": 774083.78, "VNR_MEDIANA_TCC": 281517.62},
    {"TI": 38, "EMPRESA": "ENEL SP", "VOC_MEDIA_TCC": 448533.86, "VOC_MEDIANA_TCC": 103913.50, "VNR_MEDIA_TCC": 201238.44, "VNR_MEDIANA_TCC": 180006.00},
    {"TI": 38, "EMPRESA": "Equatorial Para", "VOC_MEDIA_TCC": 2252516.51, "VOC_MEDIANA_TCC": 102720.48, "VNR_MEDIA_TCC": 2367236.17, "VNR_MEDIANA_TCC": 101517.43},
    {"TI": 38, "EMPRESA": "CPFL Piratininga", "VOC_MEDIA_TCC": 587515.01, "VOC_MEDIANA_TCC": 148692.94, "VNR_MEDIA_TCC": 594792.03, "VNR_MEDIANA_TCC": 149066.80},
])

# 2. CARREGAMENTO DA BASE CONSOLIDADA
ROOT_DIR = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_CSV = ROOT_DIR / "data" / "processed" / "base_consolidada.csv"

if not DATA_CSV.exists():
    raise FileNotFoundError(f"Base consolidada não encontrada em: {DATA_CSV.resolve()}")

df_base = pd.read_csv(DATA_CSV)

# -------------------------------------------------------------------------
# 3. MOTOR DE PROCESSAMENTO DE CENÁRIOS
# -------------------------------------------------------------------------
def gerar_estatisticas_cenario(df_sub, cenario_nome, agrupado_por_projeto=False):
    """Calcula Média e Mediana de VOC e VNR por Empresa."""
    if df_sub.empty:
        return pd.DataFrame()

    df_proc = df_sub.copy()
    if agrupado_por_projeto:
        # Chave unificada de projeto (PEP se existir, senão ODI)
        col_pep = "PEP" if "PEP" in df_proc.columns else None
        col_odi = "ODI" if "ODI" in df_proc.columns else None
        
        if col_pep and col_odi:
            df_proc["PROJETO"] = df_proc[col_pep].fillna(df_proc[col_odi])
        elif col_pep:
            df_proc["PROJETO"] = df_proc[col_pep]
        elif col_odi:
            df_proc["PROJETO"] = df_proc[col_odi]
        else:
            df_proc["PROJETO"] = np.nan

        df_proc = df_proc.groupby(["EMPRESA", "PROJETO"], as_index=False)[["VOC", "VNR"]].sum()

    resultados = []
    for emp, sub in df_proc.groupby("EMPRESA"):
        resultados.append({
            "CENARIO": cenario_nome,
            "EMPRESA": emp,
            "VOC_MEDIA_CALC": sub["VOC"].mean(),
            "VOC_MEDIANA_CALC": sub["VOC"].median(),
            "VNR_MEDIA_CALC": sub["VNR"].mean(),
            "VNR_MEDIANA_CALC": sub["VNR"].median(),
        })

    return pd.DataFrame(resultados)


def auditar_ti(ti_num: int):
    """Executa todos os cenários para uma TI e compara com a Tabela Verdade do TCC."""
    df_ti = df_base[df_base["TI"] == ti_num].copy()
    bench_ti = BENCHMARK_TCC[BENCHMARK_TCC["TI"] == ti_num]

    if df_ti.empty or bench_ti.empty:
        print(f"TI {ti_num} sem dados para auditoria.")
        return

    # Construção dos 6 Cenários
    df_pos = df_ti[(df_ti["VOC"] > 0) & (df_ti["VNR"] > 0)]
    q_low, q_high = df_ti["VOC"].quantile(0.01), df_ti["VOC"].quantile(0.99)
    df_trimmed = df_ti[(df_ti["VOC"] >= q_low) & (df_ti["VOC"] <= q_high)]

    cenarios = [
        gerar_estatisticas_cenario(df_ti, "1. Atual (Nível TUC / Ativo)", agrupado_por_projeto=False),
        gerar_estatisticas_cenario(df_pos, "2. Apenas VOC > 0 e VNR > 0", agrupado_por_projeto=False),
        gerar_estatisticas_cenario(df_ti, "4. Agrupado por Projeto (PEP/ODI)", agrupado_por_projeto=True),
        gerar_estatisticas_cenario(df_pos, "5. Projeto + (VOC > 0 e VNR > 0)", agrupado_por_projeto=True),
        gerar_estatisticas_cenario(df_trimmed, "6. Trimmed (Percentis 1-99%)", agrupado_por_projeto=False),
    ]

    df_calc_all = pd.concat(cenarios, ignore_index=True)

    # Merge com o Benchmark
    df_merged = pd.merge(df_calc_all, bench_ti, on="EMPRESA")

    # Cálculo do Erro Percentual Absoluto Médio (MAPE) por Cenário
    df_merged["ERRO_VOC_MEDIA_%"] = np.abs((df_merged["VOC_MEDIA_CALC"] - df_merged["VOC_MEDIA_TCC"]) / df_merged["VOC_MEDIA_TCC"]) * 100
    df_merged["ERRO_VOC_MEDIANA_%"] = np.abs((df_merged["VOC_MEDIANA_CALC"] - df_merged["VOC_MEDIANA_TCC"]) / df_merged["VOC_MEDIANA_TCC"]) * 100
    df_merged["ERRO_VNR_MEDIA_%"] = np.abs((df_merged["VNR_MEDIA_CALC"] - df_merged["VNR_MEDIA_TCC"]) / df_merged["VNR_MEDIA_TCC"]) * 100
    df_merged["ERRO_VNR_MEDIANA_%"] = np.abs((df_merged["VNR_MEDIANA_CALC"] - df_merged["VNR_MEDIANA_TCC"]) / df_merged["VNR_MEDIANA_TCC"]) * 100

    df_merged["ERRO_MEDIO_GLOBAL_%"] = df_merged[
        ["ERRO_VOC_MEDIA_%", "ERRO_VOC_MEDIANA_%", "ERRO_VNR_MEDIA_%", "ERRO_VNR_MEDIANA_%"]
    ].mean(axis=1)

    # Ranking de Aderência dos Cenários
    ranking = df_merged.groupby("CENARIO")["ERRO_MEDIO_GLOBAL_%"].mean().reset_index()
    ranking = ranking.sort_values(by="ERRO_MEDIO_GLOBAL_%").reset_index(drop=True)

    print(f"\n==========================================================================")
    print(f"          DIAGNÓSTICO AUTOMÁTICO DE PARIDADE - TI {ti_num}")
    print(f"==========================================================================")
    
    vencedor = ranking.iloc[0]
    if vencedor["ERRO_MEDIO_GLOBAL_%"] < 0.1:
        print(f"✔ CENÁRIO VENCEDOR: [{vencedor['CENARIO']}] - PARIDADE EXATA DE 100% (Erro: {vencedor['ERRO_MEDIO_GLOBAL_%']:.4f}%)")
    else:
        print(f"⚠ CENÁRIO MAIS PRÓXIMO: [{vencedor['CENARIO']}] - Erro Médio: {vencedor['ERRO_MEDIO_GLOBAL_%']:.2f}%")

    print("\nRanking de Aderência dos Cenários:")
    display(ranking)


# -------------------------------------------------------------------------
# 4. EXECUÇÃO AUTOMÁTICA DA AUDITORIA PARA TODAS AS TIs DO TCC
# -------------------------------------------------------------------------
for ti in [31, 33, 38]:
    auditar_ti(ti)


          DIAGNÓSTICO AUTOMÁTICO DE PARIDADE - TI 31
⚠ CENÁRIO MAIS PRÓXIMO: [5. Projeto + (VOC > 0 e VNR > 0)] - Erro Médio: 32.76%

Ranking de Aderência dos Cenários:


,CENARIO,ERRO_MEDIO_GLOBAL_%
0,5. Projeto + (VOC > 0 e VNR > 0),32.759238
1,4. Agrupado por Projeto (PEP/ODI),32.759238
2,2. Apenas VOC > 0 e VNR > 0,56.589224
3,6. Trimmed (Percentis 1-99%),59.914847
4,1. Atual (Nível TUC / Ativo),60.323587



          DIAGNÓSTICO AUTOMÁTICO DE PARIDADE - TI 33
⚠ CENÁRIO MAIS PRÓXIMO: [2. Apenas VOC > 0 e VNR > 0] - Erro Médio: 58.86%

Ranking de Aderência dos Cenários:


,CENARIO,ERRO_MEDIO_GLOBAL_%
0,2. Apenas VOC > 0 e VNR > 0,58.862153
1,1. Atual (Nível TUC / Ativo),61.346501
2,6. Trimmed (Percentis 1-99%),66.087117
3,5. Projeto + (VOC > 0 e VNR > 0),236.342552
4,4. Agrupado por Projeto (PEP/ODI),236.345435



          DIAGNÓSTICO AUTOMÁTICO DE PARIDADE - TI 38
⚠ CENÁRIO MAIS PRÓXIMO: [2. Apenas VOC > 0 e VNR > 0] - Erro Médio: 47.43%

Ranking de Aderência dos Cenários:


,CENARIO,ERRO_MEDIO_GLOBAL_%
0,2. Apenas VOC > 0 e VNR > 0,47.434562
1,1. Atual (Nível TUC / Ativo),48.053268
2,6. Trimmed (Percentis 1-99%),52.284291
3,4. Agrupado por Projeto (PEP/ODI),176.675317
4,5. Projeto + (VOC > 0 e VNR > 0),176.675317
